# 2 - Train and evaluate WSI classifiers

In [ ]:
import digitalhub as dh

In [ ]:
project = dh.get_or_create_project("cflv-1")

## Input cohorts

Each of the four artifacts must contain `labels.csv` and `20x_512px_0px_overlap/features_conch_v15/*.h5` (obtained in step 1).

Labels need `slide_name` (feature filename without extension) and integer `slide_label` columns.

Set `upload_local_cohorts=True` to log local folders, or leave it false to retrieve existing step-one artifacts. Edit the artifact names to match your project. The three additional split artifacts must exist before running training.

In [ ]:
classifier_preset = "abmil"  # abmil, mean, max, prism, titan

classifier_presets = {
    "abmil": ("patch", "abmil", "20x_512px_0px_overlap/features_conch_v15"),
    "mean": ("patch", "mean", "20x_512px_0px_overlap/features_conch_v15"),
    "max": ("patch", "max", "20x_512px_0px_overlap/features_conch_v15"),
    "prism": ("slide", "abmil", "20x_224px_0px_overlap/slide_features_prism"),
    "titan": ("slide", "abmil", "20x_512px_0px_overlap/slide_features_titan"),
}
mode, tile_pooling, feature_subdir = classifier_presets[classifier_preset]

In [ ]:
artifact_names = {
    "train": "1-trident-features-train",
    "val": "1-trident-features-val",
    "internal_test": "1-trident-features-internal-test",
    "external_test": "1-trident-features-external-test",
}

In [ ]:
cohorts = {}
for split, name in artifact_names.items():
    cohorts[split] = project.get_artifact(name)
print(split, cohorts[split].key)

## Declaration

In [ ]:
abmil_function = project.new_function(
    name="2-abmil-training",
    kind="python",
    python_version="PYTHON3_11",
    requirements="2-abmil/requirements.txt",
    code_src="2-abmil",
    handler="main:train_abmil",
    # image=abmil_image_ref
)

In [ ]:
abmil_function = project.get_function("2-abmil-training",
    entity_id="76fbd9d552f342c68b8e1b82dc97e869"
)

## Building

In [ ]:
abmil_build = abmil_function.run(
    action="build",
    instructions=[
        "apt-get update && apt-get install -y git libopenslide0 libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*"
    ],
    resources={"cpu": "8", "mem": "16Gi", "disk": "20Gi"},
)
print(abmil_build.key)

In [ ]:
abmil_image_ref = abmil_function.refresh().spec.image

## Running

Select an available *one-GPU* profile

In [ ]:
dh.get_k8s_resource_profiles()

In [ ]:
gpu_profile = '1xl40s'
output_artifact_name = "2-abmil-conch-v15-results" if classifier_preset == "abmil" else f"2-{classifier_preset}-results"
training_parameters = {
    "output_artifact_name": output_artifact_name,
    "feature_subdir": feature_subdir,
    "mode": mode,
    "tile_pooling": tile_pooling,
    "num_patches": 2000,
    "seed": 1234,
    "run_name": ("patch_features_conch_v15_abmil_np2000_h4_hd256_hid512_do02"
                 if classifier_preset == "abmil" else f"{mode}_{classifier_preset}_example"),
    # Match the upstream README training example:
    "epochs": 40,
    "patience": 8,
    "batch_size": 8,
    "eval_batch_size": 16,
    "num_workers": 4,
    "lr": 0.0003,
    "weight_decay": 0.0001,
    "hidden_dim": 512,
    "dropout": 0.2,
    "n_heads": 4,
    "head_dim": 256,
    "use_scheduler": True,
    "no_gated": False,
}

In [ ]:
abmil_run = abmil_function.run(action="job",
    inputs={
        "train_features": cohorts["train"].key,
        "val_features": cohorts["val"].key,
        "internal_test_features": cohorts["internal_test"].key,
        "external_test_features": cohorts["external_test"].key,
    },
    parameters=training_parameters,
    profile=gpu_profile,
    resources={"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "20Gi"},
    local_execution=False,
)
print(abmil_run.key)